# Hotel Management AI Agent

In [20]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio
import ollama
import sqlite3 
import re
from datetime import date
from openai import OpenAI


In [21]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"

openai = OpenAI(api_key=openai_api_key)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(base_url=ollama_url, api_key="ollama")

MODEL = "openai/gpt-oss-120b"
DB = "hotel.db"

Database

In [22]:
def create_customers_table():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS customers (
                customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
                full_name TEXT NOT NULL,
                phone TEXT NOT NULL,
                email TEXT,
                room_number TEXT NOT NULL,
                check_in TEXT NOT NULL,
                check_out TEXT NOT NULL,
                special_requests TEXT
            )
        """)
        conn.commit()

create_customers_table()

check the table is exists

In [ ]:
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    print(cursor.fetchall())

[('customers',), ('sqlite_sequence',)]


Tool functions (validation, save, view, update, delete)

Validation (validate the customer data)

In [24]:
def validate_customer_data(full_name=None, phone=None, email=None,
                            room_number=None, check_in=None, check_out=None):
    """Returns (True, None) if valid, or (False, 'reason') if not.
    Only checks fields that are provided (not None)."""

    if full_name is not None and full_name.strip() == "":
        return False, "Full name cannot be empty."

    if phone is not None:
        if not re.fullmatch(r"\+?\d{9,15}", phone):
            return False, "Phone number must be 9 to 15 digits, optionally starting with +."

    if email is not None and email.strip() != "":
        if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", email):
            return False, "Email address doesn't look valid."

    if room_number is not None and room_number.strip() == "":
        return False, "Room number cannot be empty."

    if check_in is not None:
        try:
            date.fromisoformat(check_in)
        except ValueError:
            return False, "Check-in date must be in YYYY-MM-DD format."

    if check_out is not None:
        try:
            date.fromisoformat(check_out)
        except ValueError:
            return False, "Check-out date must be in YYYY-MM-DD format."

    if check_in is not None and check_out is not None:
        if date.fromisoformat(check_out) <= date.fromisoformat(check_in):
            return False, "Check-out date must be after the check-in date."

    return True, None

Save customer data

In [25]:
def save_customer(full_name, phone, room_number, check_in, check_out,
                   email=None, special_requests=None):
    valid, error = validate_customer_data(
        full_name=full_name, phone=phone, email=email,
        room_number=room_number, check_in=check_in, check_out=check_out
    )
    if not valid:
        return {"success": False, "error": error}

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO customers (full_name, phone, email, room_number,
                                    check_in, check_out, special_requests)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (full_name, phone, email, room_number, check_in, check_out, special_requests))
        conn.commit()
        new_id = cursor.lastrowid

    return {"success": True, "customer_id": new_id}

In [26]:
print(save_customer("Jane Smith", "0712345678", "204", "2026-10-01", "2026-10-04"))

{'success': True, 'customer_id': 1}


In [31]:
# Save another, to test "multiple matches" later
print(save_customer("John Smith", "0771234567", "305", "2026-10-02", "2026-10-05"))
print(save_customer("Amara Perera", "+94712223344", "A12", "2026-11-01", "2026-11-03", email="amara@example.com"))

{'success': True, 'customer_id': 2}
{'success': True, 'customer_id': 3}


In [39]:
# Invalid input, rejected
print(save_customer("Bad Phone", "123", "101", "2026-10-01", "2026-10-04"))
# Expect: success False, error about phone digits

print(save_customer("Bad Dates", "0712345678", "101", "2026-10-05", "2026-10-01"))
# Expect: success False, error about check-out after check-in

{'success': False, 'error': 'Phone number must be 9 to 15 digits, optionally starting with +.'}
{'success': False, 'error': 'Check-out date must be after the check-in date.'}


View (Search customer data)

In [27]:
def view_customer(customer_id=None, name=None):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        if customer_id is not None:
            cursor.execute("SELECT * FROM customers WHERE customer_id = ?", (customer_id,))
        elif name is not None:
            cursor.execute("SELECT * FROM customers WHERE full_name LIKE ?", (f"%{name}%",))
        else:
            return {"success": False, "error": "Provide a customer_id or a name to search."}

        columns = [d[0] for d in cursor.description]
        rows = cursor.fetchall()

    if not rows:
        return {"success": False, "error": "No matching customer found."}

    results = [dict(zip(columns, row)) for row in rows]

    if len(results) > 1:
        return {"success": True, "multiple_matches": True, "results": results}

    return {"success": True, "multiple_matches": False, "result": results[0]}

In [ ]:
# view by id
print(view_customer(customer_id=1))

{'success': True, 'multiple_matches': False, 'result': {'customer_id': 1, 'full_name': 'Jane Smith', 'phone': '0712345678', 'email': None, 'room_number': '204', 'check_in': '2026-10-01', 'check_out': '2026-10-04', 'special_requests': None}}


In [37]:
# view by name
print(view_customer(name="Jane"))

{'success': True, 'multiple_matches': False, 'result': {'customer_id': 1, 'full_name': 'Jane Smith', 'phone': '0712345678', 'email': None, 'room_number': '204', 'check_in': '2026-10-01', 'check_out': '2026-10-04', 'special_requests': None}}


In [36]:
# View, not found
print(view_customer(customer_id=99))

{'success': False, 'error': 'No matching customer found.'}


Update 

In [29]:
def update_customer(customer_id, full_name=None, phone=None, email=None,
                     room_number=None, check_in=None, check_out=None,
                     special_requests=None):
    existing = view_customer(customer_id=customer_id)
    if not existing["success"]:
        return {"success": False, "error": "Customer not found."}

    current = existing["result"]

    # Validate against the merged (existing + new) values so cross-field
    # rules like check_out > check_in still hold
    merged_check_in = check_in if check_in is not None else current["check_in"]
    merged_check_out = check_out if check_out is not None else current["check_out"]

    valid, error = validate_customer_data(
        full_name=full_name, phone=phone, email=email,
        room_number=room_number, check_in=merged_check_in, check_out=merged_check_out
    )
    if not valid:
        return {"success": False, "error": error}

    fields = {
        "full_name": full_name, "phone": phone, "email": email,
        "room_number": room_number, "check_in": check_in,
        "check_out": check_out, "special_requests": special_requests
    }
    changes = {k: v for k, v in fields.items() if v is not None}

    if not changes:
        return {"success": False, "error": "No fields provided to update."}

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        set_clause = ", ".join(f"{k} = ?" for k in changes)
        cursor.execute(
            f"UPDATE customers SET {set_clause} WHERE customer_id = ?",
            (*changes.values(), customer_id)
        )
        conn.commit()

    return {"success": True, "customer_id": customer_id, "changed_fields": changes}

In [38]:
# update customer data
print(update_customer(customer_id=2, room_number="300"))
# Expect: success True, changed_fields: {'room_number': '305'}
print(view_customer(customer_id=2))  # confirm the change actually persisted

{'success': True, 'customer_id': 2, 'changed_fields': {'room_number': '300'}}
{'success': True, 'multiple_matches': False, 'result': {'customer_id': 2, 'full_name': 'John Smith', 'phone': '0771234567', 'email': None, 'room_number': '300', 'check_in': '2026-10-02', 'check_out': '2026-10-05', 'special_requests': None}}


Delete

In [30]:
def delete_customer(customer_id):
    existing = view_customer(customer_id=customer_id)
    if not existing["success"]:
        return {"success": False, "error": "Customer not found."}

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM customers WHERE customer_id = ?", (customer_id,))
        conn.commit()

    return {"success": True, "customer_id": customer_id}

In [40]:
print(delete_customer(customer_id=3))
print(view_customer(customer_id=3))
# Expect: deleted successfully, then "No matching customer found."

{'success': True, 'customer_id': 3}
{'success': False, 'error': 'No matching customer found.'}
